# First Attempt

In [1]:
import torch
import numpy as np

## Load/ Preprocess Data

In [ ]:
import pandas as pd
import numpy as np
import wfdb
import ast

#provided function to load data with information in the agg_df (modified to return a numpy array)
def load_raw_data(df, sampling_rate, path):
    if sampling_rate == 100:
        data = [wfdb.rdsamp(path+f) for f in df.filename_lr]
    else:
        data = [wfdb.rdsamp(path+f) for f in df.filename_hr]
    data = np.array([signal for signal, meta in data]).astype(np.float32)
    return data

# path to dta and selected sampling rate
path = "physionet.org/files/ptb-xl/1.0.3/"
sampling_rate=100

# load and convert annotation data
Y = pd.read_csv(path+'ptbxl_database.csv', index_col='ecg_id')
Y.scp_codes = Y.scp_codes.apply(lambda x: ast.literal_eval(x)) #changes class distribution to dict

# Load scp_statements.csv for diagnostic aggregation
agg_df = pd.read_csv(path+'scp_statements.csv', index_col=0)
agg_df = agg_df[agg_df.diagnostic == 1] #only take diagnostic scps

classes = agg_df.index.tolist()

# encodes the labeled as a 44d vector of 1's and zeros (based off of classes list)
def encode_multilabel(class_list):
    return np.array([1 if cls in class_list else 0 for cls in classes])

def aggregate_diagnostic(y_dic):
    tmp = []
    for key in y_dic.keys():
        if key in agg_df.index:
            if y_dic[key] > 50:
                tmp.append(agg_df.loc[key].index)
    return list(set(tmp))

# Apply diagnostic superclass and encoding multilabel
Y['diagnostic_superclass'] = Y.scp_codes.apply(aggregate_diagnostic).apply(encode_multilabel)




In [3]:
agg_df.index

Index(['NDT', 'NST_', 'DIG', 'LNGQT', 'NORM', 'IMI', 'ASMI', 'LVH', 'LAFB',
       'ISC_', 'IRBBB', '1AVB', 'IVCD', 'ISCAL', 'CRBBB', 'CLBBB', 'ILMI',
       'LAO/LAE', 'AMI', 'ALMI', 'ISCIN', 'INJAS', 'LMI', 'ISCIL', 'LPFB',
       'ISCAS', 'INJAL', 'ISCLA', 'RVH', 'ANEUR', 'RAO/RAE', 'EL', 'WPW',
       'ILBBB', 'IPLMI', 'ISCAN', 'IPMI', 'SEHYP', 'INJIN', 'INJLA', 'PMI',
       '3AVB', 'INJIL', '2AVB'],
      dtype='str')

In [4]:
# Further example code 
"""
Y_s = Y.head(10).copy()

# Load raw signal data
X = load_raw_data(Y_s, sampling_rate, path)

# Split data into train and test
test_fold = 10
# Train
X_train = X[np.where(Y_s.strat_fold != test_fold)]
y_train = Y_s[(Y_s.strat_fold != test_fold)].diagnostic_superclass
# Test
X_test = X[np.where(Y_s.strat_fold == test_fold)]
y_test = Y_s[Y_s.strat_fold == test_fold].diagnostic_superclass"""

'\nY_s = Y.head(10).copy()\n\n# Load raw signal data\nX = load_raw_data(Y_s, sampling_rate, path)\n\n# Split data into train and test\ntest_fold = 10\n# Train\nX_train = X[np.where(Y_s.strat_fold != test_fold)]\ny_train = Y_s[(Y_s.strat_fold != test_fold)].diagnostic_superclass\n# Test\nX_test = X[np.where(Y_s.strat_fold == test_fold)]\ny_test = Y_s[Y_s.strat_fold == test_fold].diagnostic_superclass'

In [5]:
#Y_s.diagnostic_superclass.to_numpy() #example line run of multilabel distribution
#Y['scp_codes']

In [6]:
Y['scp_codes']

ecg_id
1                 {'NORM': 100.0, 'LVOLT': 0.0, 'SR': 0.0}
2                             {'NORM': 80.0, 'SBRAD': 0.0}
3                               {'NORM': 100.0, 'SR': 0.0}
4                               {'NORM': 100.0, 'SR': 0.0}
5                               {'NORM': 100.0, 'SR': 0.0}
                               ...                        
21833    {'NDT': 100.0, 'PVC': 100.0, 'VCLVH': 0.0, 'ST...
21834             {'NORM': 100.0, 'ABQRS': 0.0, 'SR': 0.0}
21835                           {'ISCAS': 50.0, 'SR': 0.0}
21836                           {'NORM': 100.0, 'SR': 0.0}
21837                           {'NORM': 100.0, 'SR': 0.0}
Name: scp_codes, Length: 21799, dtype: object

## Data Prep

In [7]:
from sklearn.model_selection import train_test_split

Y_s = Y.sample(100)

Y_transformed = np.stack(Y_s.diagnostic_superclass.values).astype(np.float32)

# Load raw signal data
X = load_raw_data(Y_s, sampling_rate, path)

X = np.transpose(X, (0, 2, 1))

X_train, X_test, y_train, y_test = train_test_split(X, Y_transformed, test_size=0.1, random_state=56)

In [8]:
X_train_torch = torch.tensor(X_train) # change dim ordering for PyTorch
y_train_torch = torch.tensor(y_train)
X_test_torch = torch.tensor(X_test) # change dim ordering for PyTorch
y_test_torch = torch.tensor(y_test)

In [9]:
train_dataset = torch.utils.data.TensorDataset(X_train_torch, y_train_torch)
train_dataloader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=32, shuffle=True)

test_dataset = torch.utils.data.TensorDataset(X_test_torch, y_test_torch)
test_dataloader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size=32)


## CNN

In [10]:
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F


class CNet(nn.Module):
  def __init__(self):
    super(CNet,self).__init__()
    #starting with 96

    self.conv11 = nn.Conv1d(12, 64, 9, padding=4)
    self.conv12 = nn.Conv1d(64, 64, 9, padding=4)
    self.bn1 = nn.BatchNorm1d(64)

    self.pool = nn.MaxPool1d(kernel_size=2, stride = 2)

    self.conv21 = nn.Conv1d(64, 128, 9, padding=4)
    self.conv22 = nn.Conv1d(128, 128, 9, padding=4)
    self.bn2 = nn.BatchNorm1d(128)

    self.conv31 = nn.Conv1d(128, 256, 9, padding=4)
    self.conv32 = nn.Conv1d(256, 256, 9, padding=4)
    self.conv33 = nn.Conv1d(256, 256, 9, padding=4)
    self.bn3 = nn.BatchNorm1d(256)

    self.conv41 = nn.Conv1d(256, 512, 9, padding=4)
    self.conv42 = nn.Conv1d(512, 512, 9, padding=4)
    self.conv43 = nn.Conv1d(512, 512, 9, padding=4)
    self.bn4 = nn.BatchNorm1d(512)

    self.fc1 = nn.Linear(512 * 125, 1000)
    self.fc2 = nn.Linear(1000, 500)
    self.fc3 = nn.Linear(500, 44)

  def forward(self, x):
    #x = self.pool(F.relu(self.conv12(F.relu(self.conv11(x)))))
    #x = self.pool(F.relu(self.conv22(F.relu(self.conv21(x)))))
    #x = self.pool(F.relu(self.conv33(F.relu(self.conv32(F.relu(self.conv31(x)))))))
    #x = F.relu(self.conv43(F.relu(self.conv42(F.relu(self.conv41(x))))))

    x = self.pool(F.relu(self.bn1(self.conv12(F.relu(self.conv11(x))))))
    x = self.pool(F.relu(self.bn2(self.conv22(F.relu(self.conv21(x))))))
    x = self.pool(F.relu(self.bn3(self.conv33(F.relu(self.conv32(F.relu(self.conv31(x))))))))
    x = F.relu(self.bn4(self.conv43(F.relu(self.conv42(F.relu(self.conv41(x)))))))
    x = torch.flatten(x,1)
    x = F.relu(self.fc1(x))
    x = F.relu(self.fc2(x))
    x = self.fc3(x)
    return x

In [11]:
def train_model(model, train_dataloader, test_dataloader, epochs, weight_decay):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    model.train()

    #loss_fn = torch.nn.CrossEntropyLoss()
    loss_fn = nn.BCEWithLogitsLoss()

    lr = 1e-3
    opt = torch.optim.SGD(model.parameters(), lr=lr, weight_decay=weight_decay)
    #opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    #scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    best_test = 0
    best_test_train = 0
    best_epoch = 0

    for epoch in range(epochs):
        running_loss = 0
        for batch_index, (X, y) in enumerate(train_dataloader):
            print(f"Epoch: {epoch} Batch {batch_index}")
            X,y = X.to(device), y.to(device)

            opt.zero_grad() #zero out the gradients

            z = model(X)
            loss = loss_fn(z,y) #compute loss
            running_loss += loss.item()

            loss.backward() #compute gradients

            opt.step() #apply gradients
            #scheduler.step()



        running_loss = running_loss / len(train_dataloader)
        print(f"Epoch {epoch} train loss: {running_loss}")


In [12]:
X.shape

(100, 12, 1000)

In [ ]:
def jaccard_score(y_pred, y_true):
    intersection = ((y_pred == 1) & (y_true == 1)).sum()
    union = ((y_pred == 1) | (y_true == 1)).sum()

    if union == 0:
        return 0.0
    
    return intersection / union

def precision(y_pred, y_true):
    f_p = ((y_pred == 1) & (y_true == 1)).sum()
    t_p = ((y_pred == 1) & (y_true == 0)).sum()
    if t_p + f_p == 0:
        return 0.0
    return t_p / (t_p + f_p)

def recall(y_pred, y_true):
    t_p = ((y_pred == 1) & (y_true == 1)).sum()
    f_n = ((y_pred == 0) & (y_true == 1)).sum()

    if t_p + f_n == 0:
        return 0.0
    return t_p / (t_p + f_n)

def f1(y_pred, y_true):
    m_p = precision(y_pred, y_true)
    m_r = recall(y_pred, y_true)
    if m_p + m_r == 0:
        return 0.0
    return 2 * (m_p*m_r) / (m_p+m_r)

def

In [23]:
y

tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])

In [13]:
net = CNet()
train_model(net, train_dataloader, test_dataloader, 30, 1e-2)

Epoch: 0 Batch 0
Epoch: 0 Batch 1
Epoch: 0 Batch 2
Epoch 0 train loss: 0.6968211929003397
Epoch: 1 Batch 0
Epoch: 1 Batch 1
Epoch: 1 Batch 2
Epoch 1 train loss: 0.693821648756663
Epoch: 2 Batch 0
Epoch: 2 Batch 1
Epoch: 2 Batch 2
Epoch 2 train loss: 0.691437820593516
Epoch: 3 Batch 0
Epoch: 3 Batch 1
Epoch: 3 Batch 2
Epoch 3 train loss: 0.6887073516845703
Epoch: 4 Batch 0
Epoch: 4 Batch 1
Epoch: 4 Batch 2
Epoch 4 train loss: 0.6861081520716349
Epoch: 5 Batch 0
Epoch: 5 Batch 1
Epoch: 5 Batch 2
Epoch 5 train loss: 0.683865487575531
Epoch: 6 Batch 0
Epoch: 6 Batch 1
Epoch: 6 Batch 2
Epoch 6 train loss: 0.681289792060852
Epoch: 7 Batch 0
Epoch: 7 Batch 1
Epoch: 7 Batch 2
Epoch 7 train loss: 0.6789369781812032
Epoch: 8 Batch 0
Epoch: 8 Batch 1
Epoch: 8 Batch 2
Epoch 8 train loss: 0.676596204439799
Epoch: 9 Batch 0
Epoch: 9 Batch 1
Epoch: 9 Batch 2
Epoch 9 train loss: 0.6742985844612122
Epoch: 10 Batch 0
Epoch: 10 Batch 1
Epoch: 10 Batch 2
Epoch 10 train loss: 0.6723758776982626
Epoch: 11 B